In [ ]:
"""
sim_and_plot.py
===============
Six-panel comparison of all estimation methods.

Panels
------
  1. Autonomous    – no control
  2. Optimal       – true parameters
  3. Alg 1 batch   – NbhdVARTracker  (multi-trajectory VAR, neighbourhood filter)
  4. Alg 1 online  – OnlineVARTracker (streaming EMA-VAR, neighbourhood filter)
  5. Alg 2 batch   – CCATracker       (past-future CCA, with boundary restarts)
  6. Alg 2 online  – OnlineCCATracker (streaming EMA-CCA)

Algorithm 1 uses erf_feedback  (w_u unit vector, lambda_u continuous-time, D scalar).
Algorithm 2 uses cca_feedback  (v0 CCA-normalised, lambda0 discrete, Dv scalar).
These two controller interfaces are DIFFERENT – do not mix them.
"""

import os
import numpy as np
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt
from matplotlib.lines import Line2D
from matplotlib.collections import LineCollection
from functools import partial
import committor_control as cc

os.makedirs("figures", exist_ok=True)

# ─────────────────────────────────────────────────────────────────────────────
# Colour constants
# ─────────────────────────────────────────────────────────────────────────────
COL_DESIRED   = "#208820"
COL_UNDESIRED = "#aa0000"
COL_WU_TRUE   = "darkblue"


# ─────────────────────────────────────────────────────────────────────────────
# Low-level drawing helpers
# ─────────────────────────────────────────────────────────────────────────────

def draw_background(ax, x_fp, w_u, w_s, a, xlim, ylim):
    ax.axvspan(xlim[0], x_fp[0], color="#cce8cc", alpha=0.30, zorder=0)
    ax.axvspan(x_fp[0], xlim[1], color="#ffd9d9", alpha=0.30, zorder=0)
    for sign, c in [(-1, COL_DESIRED), (+1, COL_UNDESIRED)]:
        ctr = x_fp + sign * a * w_u
        ts  = np.linspace(-1.5, 1.5, 120)
        ln  = ctr[:, None] + np.outer(w_s, ts)
        ax.plot(ln[0], ln[1], color=c, lw=1.6, ls="--", alpha=0.85, zorder=3)
    ax.scatter(*x_fp, s=200, color="black", marker="+",
               linewidths=2.2, zorder=7)


def draw_arrow(ax, x_fp, direction, scale=0.22,
               color=COL_WU_TRUE, lw=1.8, ls="solid",
               mutation_scale=12, alpha=1.0, zorder=6):
    ax.annotate("", xy=x_fp + scale * direction, xytext=x_fp,
                arrowprops=dict(arrowstyle="-|>", color=color, lw=lw,
                                mutation_scale=mutation_scale,
                                linestyle=ls, alpha=alpha),
                zorder=zorder)


def finish_ax(ax, xlim, ylim):
    ax.set_xlim(xlim); ax.set_ylim(ylim)
    ax.set_xticks([]); ax.set_yticks([])
    ax.set_aspect("equal")


def draw_region_labels(ax, a):
    ax.text(-a + 0.03, a - 0.05, "desired",
            color=COL_DESIRED, fontsize=9, va="top", ha="left", fontweight="bold")
    ax.text(a - 0.03, a - 0.05, "undesired",
            color=COL_UNDESIRED, fontsize=9, va="top", ha="right", fontweight="bold")


# ─────────────────────────────────────────────────────────────────────────────
# Figure A: single trajectory per panel
# ─────────────────────────────────────────────────────────────────────────────

def plot_single_traj(panels, sys_dw, a,
                     seed=3, T_sim=2000, dt=0.01, D_true=0.01,
                     figsize=None):
    """
    panels : list of (label, ctrl_fn, color, est_direction_or_None)
             est_direction : unit vector drawn as a dashed arrow, or None.
    """
    w_s  = np.array([-sys_dw.w_u_true[1], sys_dw.w_u_true[0]])
    pad  = 0.06
    xlim = (-a - pad, a + pad)
    ylim = (-a - pad, a + pad)

    rng = np.random.default_rng(seed)
    x0  = (sys_dw.x_star
           + rng.uniform(0.05, a * 0.80) * sys_dw.w_u_true
           + rng.uniform(-a * 0.40, a * 0.40) * w_s)
    ns  = int(rng.integers(0, 10**6))

    n = len(panels)
    if figsize is None:
        figsize = (4.4 * n, 5.0)

    fig, axes = plt.subplots(1, n, figsize=figsize, sharey=True)
    if n == 1:
        axes = [axes]
    fig.subplots_adjust(wspace=0.04, left=0.02, right=0.99,
                        top=0.84, bottom=0.13)

    for ax, (label, ctrl, col, est_dir) in zip(axes, panels):
        draw_background(ax, sys_dw.x_star, sys_dw.w_u_true, w_s, a, xlim, ylim)
        draw_arrow(ax, sys_dw.x_star, sys_dw.w_u_true, color=COL_WU_TRUE, lw=1.8)

        if est_dir is not None:
            draw_arrow(ax, sys_dw.x_star, est_dir, scale=0.20,
                       color=col, lw=1.6, ls="dashed", mutation_scale=11)

        X, hs = cc.simulate_trajectory(
            x0, T_sim, dt, sys_dw.drift, D_true,
            control_func=ctrl,
            x_fp=sys_dw.x_star, w_u=sys_dw.w_u_true, a=a,
            rng=np.random.default_rng(ns))

        pts  = X[:, :2].reshape(-1, 1, 2)
        segs = np.concatenate([pts[:-1], pts[1:]], axis=1)
        lc   = LineCollection(segs, cmap="viridis",
                              norm=plt.Normalize(0, len(X)),
                              lw=1.5, alpha=0.85, zorder=4)
        lc.set_array(np.arange(len(X) - 1))
        ax.add_collection(lc)

        if ctrl is not None:
            step_u = max(1, len(X) // 50)
            idx_u  = np.arange(0, len(X) - 1, step_u)
            Us = np.array([ctrl(X[t]) for t in idx_u])
            u_norms = np.linalg.norm(Us, axis=1)
            u_max   = u_norms.max() if u_norms.max() > 1e-10 else 1.0
            ax.quiver(X[idx_u, 0], X[idx_u, 1],
                      Us[:, 0] * 0.06 / u_max,
                      Us[:, 1] * 0.06 / u_max,
                      u_norms / u_max,
                      cmap="YlOrRd", norm=plt.Normalize(0, 1),
                      scale=1, scale_units="xy",
                      width=0.005, headwidth=3.5, headlength=4.0,
                      alpha=0.80, zorder=5)

        ax.scatter(*X[-1], s=50, color=col, edgecolors="white", lw=0.6, zorder=6)
        outcome = ('✓ desired' if hs == 'negative' else
                   '✗ undesired' if hs == 'positive' else 'timeout')
        ax.set_title(f"{label}\n({outcome})",
                     fontsize=10, pad=4,
                     color=col if label != "Autonomous" else "#333")
        finish_ax(ax, xlim, ylim)
        if ax is axes[0]:
            draw_region_labels(ax, a)

    legend_handles = [
        Line2D([0],[0], color=COL_WU_TRUE, lw=2.0, label=r"$w_u$ (true)"),
        Line2D([0],[0], color=COL_DESIRED,   lw=1.4, ls="--", label=r"$\partial\mathcal{A}$"),
        Line2D([0],[0], color=COL_UNDESIRED, lw=1.4, ls="--", label=r"$\partial\mathcal{B}$"),
    ]
    for label, _, col, est_dir in panels:
        if est_dir is not None:
            legend_handles.append(
                Line2D([0],[0], color=col, lw=1.5, ls="--",
                       label=rf"$\hat{{w}}_u$ – {label.replace(chr(10), ' ')}"))
    fig.legend(handles=legend_handles, loc="lower center",
               ncol=len(legend_handles), fontsize=9, frameon=True,
               bbox_to_anchor=(0.5, -0.03))
    return fig


# ─────────────────────────────────────────────────────────────────────────────
# Figure B: many trajectories per panel
# ─────────────────────────────────────────────────────────────────────────────

def plot_multi_traj(panels, all_trajs, sys_dw, a, figsize=None):
    """
    panels    : list of (label, ctrl_fn, color, est_direction_or_None)
    all_trajs : dict  label -> list of (X array, hit_side str)
    """
    w_s  = np.array([-sys_dw.w_u_true[1], sys_dw.w_u_true[0]])
    pad  = 0.06
    xlim = (-a - pad, a + pad)
    ylim = (-a - pad, a + pad)

    n = len(panels)
    if figsize is None:
        figsize = (4.4 * n, 5.0)

    fig, axes = plt.subplots(1, n, figsize=figsize, sharey=True)
    if n == 1:
        axes = [axes]
    fig.subplots_adjust(wspace=0.04, left=0.02, right=0.99,
                        top=0.84, bottom=0.13)

    for ax, (label, ctrl, col, est_dir) in zip(axes, panels):
        draw_background(ax, sys_dw.x_star, sys_dw.w_u_true, w_s, a, xlim, ylim)
        draw_arrow(ax, sys_dw.x_star, sys_dw.w_u_true, color=COL_WU_TRUE, lw=1.8)

        if est_dir is not None:
            draw_arrow(ax, sys_dw.x_star, est_dir, scale=0.20,
                       color=col, lw=1.6, ls="dashed", mutation_scale=11)

        trajs = all_trajs[label]
        n_t   = len(trajs)
        hit_a = sum(1 for _, hs in trajs if hs == "negative")
        alpha = max(0.10, min(0.50, 5.0 / n_t))

        for X, _ in trajs:
            ax.plot(X[:, 0], X[:, 1], color=col, lw=1.1, alpha=alpha, zorder=4)
            ax.scatter(*X[-1], s=16, color=col, edgecolors="white", lw=0.3, zorder=5)

        ax.set_title(f"{label}\nP($\\mathcal{{A}}$) = {hit_a}/{n_t}",
                     fontsize=10, pad=5,
                     color=col if label != "Autonomous" else "#333")
        finish_ax(ax, xlim, ylim)
        if ax is axes[0]:
            draw_region_labels(ax, a)

    fig.legend(handles=[
        Line2D([0],[0], color=COL_WU_TRUE, lw=2.0, label=r"$w_u$ (true)"),
        Line2D([0],[0], color=COL_DESIRED,   lw=1.4, ls="--", label=r"$\partial\mathcal{A}$"),
        Line2D([0],[0], color=COL_UNDESIRED, lw=1.4, ls="--", label=r"$\partial\mathcal{B}$"),
    ], loc="lower center", ncol=3, fontsize=9, frameon=True,
       bbox_to_anchor=(0.5, -0.03))
    return fig


# ═════════════════════════════════════════════════════════════════════════════
# Main
# ═════════════════════════════════════════════════════════════════════════════

sys_dw      = cc.make_double_well_curved(lambda_u=2.0, lambda_s=10.0, c=0.5)
dt          = 0.01
D_true      = 0.01
a           = 0.5
r_neigh     = 0.5
T_budget    = 2500
seed_id     = 0
seed_single = 3
seed_multi  = 11
n_show      = 40
T_sim       = 2000
GAMMA       = 0.02    # EMA decay for online methods

# ── System identification ──────────────────────────────────────────────────

# ---- Algorithm 1 batch: NbhdVARTracker ----
# Multiple short trajectories, VAR regression, neighbourhood filter.
# Uses: erf_feedback(w_u, lambda_u, D, x_fp)
alg1 = cc.NbhdVARTracker(dim=2, radius=0.3, x_star_init_range=0.2,
                          store_history=True, rng_seed=seed_id)
alg1.train(sys_dw.drift, n_trajs=50, T=50, dt=dt, D=D_true,
           seed=seed_id, init_bounds=0.1, n_iterations=15)
p1 = alg1.get_params(dt, sign_ref=sys_dw.w_u_true)
# p1 keys: w_u (unit), lambda_u (continuous-time), D (scalar), x_star

# ---- Algorithm 1 online: OnlineVARTracker ----
# Single trajectory, EMA-updated VAR covariances, neighbourhood filter.
# Uses: erf_feedback(w_u, lambda_u, D, x_fp)
alg1o = cc.OnlineVARTracker(dim=2, radius=0.3, gamma=GAMMA, min_steps=50,
                              store_history=True, rng_seed=seed_id)
alg1o.train(sys_dw.drift, T=T_budget, dt=dt, D=D_true,
            seed=seed_id, sign_ref=sys_dw.w_u_true)
p1o = alg1o.get_params(dt, sign_ref=sys_dw.w_u_true)
# p1o keys: w_u (unit), lambda_u (continuous-time), D (scalar), x_star

# ---- Algorithm 2 batch: CCATracker ----
# Single trajectory with boundary restarts, past-future CCA on (x_t, x_{t+1}).
# Uses: cca_feedback(v0, lambda0, Dv, x_star)  – DIFFERENT interface from Alg 1
alg2 = cc.CCATracker(dim=2, store_history=True, rng_seed=seed_id)
alg2.train(sys_dw.drift, T=T_budget, dt=dt, D=D_true,
           seed=seed_id, init_bounds=0.1, sign_ref=sys_dw.w_u_true,
           a=a, init_spread=0.05)
p2 = alg2.get_params(sign_ref=sys_dw.w_u_true)
# p2 keys: v0 (C_pp-normalised), lambda0 (discrete), Dv (v0^T D v0), Delta, x_star

# ---- Algorithm 2 online: OnlineCCATracker ----
# Single trajectory, EMA-updated CCA covariances.
# Uses: cca_feedback(v0, lambda0, Dv, x_star)  – DIFFERENT interface from Alg 1
alg2o = cc.OnlineCCATracker(dim=2, radius=None, gamma=GAMMA, min_steps=50,
                              a=a, init_spread=0.05,
                              store_history=True, rng_seed=seed_id)
alg2o.train(sys_dw.drift, T=T_budget, dt=dt, D=D_true,
            seed=seed_id, sign_ref=sys_dw.w_u_true)
p2o = alg2o.get_params(sign_ref=sys_dw.w_u_true)
# p2o keys: v0 (C_pp-normalised), lambda0 (discrete), Dv (v0^T D v0), Delta, x_star

# ── Parameter table ────────────────────────────────────────────────────────
v0_unit  = p2['v0']  / np.linalg.norm(p2['v0'])
v0o_unit = p2o['v0'] / np.linalg.norm(p2o['v0'])

print(f"Parameter estimates  (T_budget={T_budget},  γ={GAMMA})")
print(f"{'':14}{'True':>9}  {'Alg1 batch':>11}  {'Alg1 online':>12}  "
      f"{'Alg2 batch':>11}  {'Alg2 online':>12}")
print(f"{'Method':14}{'':>9}  {'NbhdVAR':>11}  {'EMA-VAR':>12}  "
      f"{'CCA+R':>11}  {'EMA-CCA':>12}")

rows = [
    ("w_u[0]",
     sys_dw.w_u_true[0],
     p1['w_u'][0],   p1o['w_u'][0],  v0_unit[0],   v0o_unit[0]),
    ("w_u[1]",
     sys_dw.w_u_true[1],
     p1['w_u'][1],   p1o['w_u'][1],  v0_unit[1],   v0o_unit[1]),
    ("λ_disc",
     np.exp(sys_dw.lambda_u_true * dt),
     np.exp(p1['lambda_u'] * dt), np.exp(p1o['lambda_u'] * dt),
     p2['lambda0'],                p2o['lambda0']),
    ("r=λ/(2D)",
     sys_dw.lambda_u_true / (2 * D_true),
     p1['lambda_u']  / (2 * p1['D']),
     p1o['lambda_u'] / (2 * p1o['D']),
     p2['lambda0']  / max(4 * p2['Dv'],  1e-15),
     p2o['lambda0'] / max(4 * p2o['Dv'], 1e-15)),
    ("x*[0]", 0.0,
     p1['x_star'][0], p1o['x_star'][0], p2['x_star'][0], p2o['x_star'][0]),
    ("x*[1]", 0.0,
     p1['x_star'][1], p1o['x_star'][1], p2['x_star'][1], p2o['x_star'][1]),
]
for nm, tv, v1, v1o, v2, v2o in rows:
    print(f"  {nm:<12}{tv:>9.4f}  {v1:>11.4f}  {v1o:>12.4f}  {v2:>11.4f}  {v2o:>12.4f}")

# ── Controllers ───────────────────────────────────────────────────────────
kw_erf = dict(a=a, r_neigh=r_neigh, nbhd_shape="circular")

ctrl_opt = partial(cc.erf_feedback,
                   w_u=sys_dw.w_u_true, lambda_u=sys_dw.lambda_u_true,
                   D=D_true, x_fp=sys_dw.x_star, **kw_erf)

# Algorithm 1 controllers – erf_feedback
ctrl_alg1  = partial(cc.erf_feedback,
                     w_u=p1['w_u'],  lambda_u=p1['lambda_u'],
                     D=p1['D'],  x_fp=p1['x_star'],  **kw_erf)
ctrl_alg1o = partial(cc.erf_feedback,
                     w_u=p1o['w_u'], lambda_u=p1o['lambda_u'],
                     D=p1o['D'], x_fp=p1o['x_star'], **kw_erf)

# Algorithm 2 controllers – cca_feedback  (DIFFERENT interface)
ctrl_alg2  = partial(cc.cca_feedback,
                     v0=p2['v0'],  lambda0=p2['lambda0'],
                     Dv=p2['Dv'], x_star=p2['x_star'],
                     a=a, r_neigh=r_neigh)
ctrl_alg2o = partial(cc.cca_feedback,
                     v0=p2o['v0'], lambda0=p2o['lambda0'],
                     Dv=p2o['Dv'], x_star=p2o['x_star'],
                     a=a, r_neigh=r_neigh)

# ── Panel definitions ─────────────────────────────────────────────────────
#    (label, ctrl, colour, est_direction_for_arrow)
#    Alg 1 arrow: unit w_u
#    Alg 2 arrow: unit v_0 (normalised from CCA)
panels = [
    ("Autonomous",             None,        "#d62728", None),
    ("Optimal",                ctrl_opt,    "#1a6faf", None),
    ("Algorithm 1\n(batch)",   ctrl_alg1,   "#e07b00", p1['w_u']),
    ("Algorithm 1\n(online)",  ctrl_alg1o,  "#ff7f0e", p1o['w_u']),
    ("Algorithm 2\n(batch)",   ctrl_alg2,   "#2ca02c", v0_unit),
    ("Algorithm 2\n(online)",  ctrl_alg2o,  "#17becf", v0o_unit),
]

# ── Simulate for multi-traj figure ──────────────────────────────────────────
w_s    = np.array([-sys_dw.w_u_true[1], sys_dw.w_u_true[0]])
rng_ic = np.random.default_rng(seed_multi)
ics    = [sys_dw.x_star
          + rng_ic.uniform(0.02, a * 0.9) * sys_dw.w_u_true
          + rng_ic.uniform(-a * 0.5, a * 0.5) * w_s
          for _ in range(n_show)]
nseeds = [int(rng_ic.integers(0, 10**6)) for _ in range(n_show)]

all_trajs = {}
for label, ctrl, _, _ in panels:
    trajs = []
    for x0_ic, ns in zip(ics, nseeds):
        X, hs = cc.simulate_trajectory(
            x0_ic, T_sim, dt, sys_dw.drift, D_true,
            control_func=ctrl,
            x_fp=sys_dw.x_star, w_u=sys_dw.w_u_true, a=a,
            rng=np.random.default_rng(ns))
        trajs.append((X, hs))
    all_trajs[label] = trajs

print("\nClosed-loop P(reaching desired set A):")
for label, _, _, _ in panels:
    hit_a = sum(1 for _, hs in all_trajs[label] if hs == "negative")
    print(f"  {label.replace(chr(10), ' '):<32} P(A) = {hit_a}/{n_show}")

# ── Figure A: single trajectory ───────────────────────────────────────────
suptitle = (r"Curved double-well  ($D{=}0.01$,  $\lambda_u{=}2$,  "
            rf"$T={T_budget}$)")
fig_a = plot_single_traj(panels, sys_dw, a,
                         seed=seed_single, T_sim=T_sim, dt=dt, D_true=D_true)
fig_a.suptitle(suptitle + "  –  single trajectory", fontsize=11, y=1.02)
fig_a.savefig("figures/single_traj_comparison.pdf", dpi=200, bbox_inches="tight")
fig_a.savefig("figures/single_traj_comparison.png", dpi=200, bbox_inches="tight")
print("\nSaved: figures/single_traj_comparison.png")

# ── Figure B: many trajectories ───────────────────────────────────────────
fig_b = plot_multi_traj(panels, all_trajs, sys_dw, a)
fig_b.suptitle(suptitle + rf"  –  {n_show} trajectories", fontsize=11, y=0.99)
fig_b.savefig("figures/multi_traj_comparison.pdf", dpi=200, bbox_inches="tight")
fig_b.savefig("figures/multi_traj_comparison.png", dpi=200, bbox_inches="tight")
print("Saved: figures/multi_traj_comparison.png")

Parameter estimates  (T_budget=2500,  γ=0.02)
                   True   Alg1 batch   Alg1 online   Alg2 batch   Alg2 online
Method                       NbhdVAR       EMA-VAR        CCA+R       EMA-CCA
  w_u[0]         1.0000       0.9971        0.9545       0.9991        0.9253
  w_u[1]         0.0000      -0.0758       -0.2982       0.0432       -0.3793
  λ_disc         1.0202       1.0192        1.0242       1.0193        0.9960
  r=λ/(2D)     100.0000      96.8948      200.2133      64.7626     3000.0448
  x*[0]          0.0000       0.0012        0.0623       0.0493       -0.6884
  x*[1]          0.0000       0.0020        0.0136       0.0201        0.3682

Closed-loop P(reaching desired set A):
  Autonomous                       P(A) = 1/40
  Optimal                          P(A) = 40/40
  Algorithm 1 (batch)              P(A) = 40/40
  Algorithm 1 (online)             P(A) = 40/40
  Algorithm 2 (batch)              P(A) = 16/40
  Algorithm 2 (online)             P(A) = 1/40

Sav